In [25]:

import mne_nirs
import mne
from mne_bids import BIDSPath, get_entity_vals
from mne.preprocessing.nirs import optical_density, temporal_derivative_distribution_repair
from itertools import compress
import numpy as np

###  The preprocess pipeline explained in the description of this tutorial is applied in this function.

In [31]:
def preprocessing(subject_index, t_min,t_max):
    
    """
    Arguments:
    Subject index: int = Index of the subject to be processed [0, 1, 2, 3, 4, 5, 6, 7]
    t_min: float =  Time to initialize the epoch
    t_max: float = Time to end the epoch
    
    Returns:
    haemo:= The preprocessed haemoglobin data
    epochs: mne.Epochs object = The preprocessed epochs. You can use this to get the data and labels for the classification task.
    X: np.array = The preprocessed data. Shape: (n_trials, n_channels, n_times)
    Y: np.array = The labels for the classification task. Shape: (n_trials,)
    Y labels : 1.0-> Audio, 3.0 -> Control (Silence)
    """
    
    root = mne_nirs.datasets.audio_or_visual_speech.data_path()
    
    print(root)
    
    subject= get_entity_vals(root, "subject")[subject_index]
   
        
    dataset = BIDSPath(
            root=root,
            suffix="nirs",
            extension=".snirf",
            subject=subject,
            task="AudioVisualBroadVsRestricted",
            datatype="nirs",
            session="01"
                            )
                
    #Raw intensity data
    raw_intensity = mne.io.read_raw_snirf(dataset.fpath)
    
    raw_intensity.annotations.rename(
        {"1.0": "Audio", "2.0": "Video", "3.0": "Control", "15.0": "Ends"}
    )
    
    
    # Converting raw intensity to optical density
    raw_od = optical_density(raw_intensity)


    #Compute scalp coupling index to identify optodes that were not well attached to the scalp
    #Rejection criterion of < 0.8
    sci = mne.preprocessing.nirs.scalp_coupling_index(raw_od)
    raw_od.info["bads"] = list(compress(raw_od.ch_names, sci < 0.8))
    
    print("Number of channels before removing bad channels: ", len(raw_od.ch_names))
    #raw_od = raw_od.info.pop('bads')

    raw_od = raw_od.drop_channels(raw_od.info['bads'])

    print("Number of channels after removing bad channels: ", len(raw_od.ch_names))
        

    
    #Temporal Derivative Distribution Repair
    
    corrected_tddr = temporal_derivative_distribution_repair(raw_od)
    
    #Apply short channel correction  to remove the influence from non-cortical changes in blood oxygenation.
    
    # A short separation channels measures solely the extracerebral signals, which includes 
    # blood presure waves, mayer waves, respiration and cardiac cycles.
    # The signal components od the short separation channel can be seen as the noise in the signal of the 
    # long channel.BY removing these components from the log channel, you cna minimize the noise.
    
    
    od_corrected = mne_nirs.signal_enhancement.short_channel_regression(corrected_tddr)
    #Convert optical density to haemoglobin concentration using the Beer-Lambert Law
    haemo = mne.preprocessing.nirs.beer_lambert_law(od_corrected, ppf=6)
    
    haemo = mne_nirs.channels.get_long_channels(haemo)
    
    #Bandpass filter the haemoglobin data between 0.02 and 0.4 Hz
    #to removoe slow drifts and components related to the heart rate
    haemo = haemo.filter(0.02,0.4)

    #plot haemo

    
    #haemo.plot(n_channels=1, duration= 32, show_scrollbars=False)
    
    
    #Signal enhancement method (negative correlation enhancement algorithm) Cui et. al. 2010
    
    haemo = mne_nirs.signal_enhancement.enhance_negative_correlation(haemo)
    

        
    events, event_dict = mne.events_from_annotations(haemo)
    #Epochs corresponding to 8 s before the stimulus onset and 30 s after the stimulus onset
    #An epoch rejection criterion  was employed to exclude epochs with a signal amplitude > 100 uM
    epochs = mne.Epochs(
        haemo,
        events,
        event_id=event_dict,
        tmin=t_min,
        tmax=t_max,
        reject=dict(hbo=100e-6, hbr=100e-6), # Epoc rejection criterion
        reject_by_annotation=True,
        proj=False,
        baseline=None,
        detrend=None,
        preload=True,
        verbose=True,
    )
    
    # I am just selecting the Audio, Video, and Control epochs
    epochs = epochs[["Audio", "Video", "Control"]]
    
    return haemo, epochs

Let's say that we will load the data from the subject one, and set an epoch from the stimulus onset to 18 s after the stimulus onset. 

In [32]:
subject_index = 0
t_min = 0.0
t_max = 18.0
haemo, epochs = preprocessing(subject_index, t_min, t_max)

/home/sposso22/mne_data/fNIRS-audio-visual-speech
Loading /home/sposso22/mne_data/fNIRS-audio-visual-speech/sub-01/ses-01/nirs/sub-01_ses-01_task-AudioVisualBroadVsRestricted_nirs.snirf
Reading 0 ... 7976  =      0.000 ...  2041.856 secs...
Number of channels before removing bad channels:  104
Number of channels after removing bad channels:  90
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.02 - 0.4 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.02
- Lower transition bandwidth: 0.02 Hz (-6 dB cutoff frequency: 0.01 Hz)
- Upper passband edge: 0.40 Hz
- Upper transition bandwidth: 1.55 Hz (-6 dB cutoff frequency: 1.18 Hz)
- Filter length: 645 samples (165.120 s)

Used Annotations descriptions: [np.str_('Audio'), np.str_('Control'), np.str_('Ends'), np

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.0s


In [28]:
#Epochs is a mne.Epochs object. You can use this to get the data and labels for each trial 

#fnirs data preprocessed 
X = epochs.get_data()
#Labels for each trial 
Y = epochs.events[:, 2]

print(X.shape)
print(Y.shape)



(46, 76, 71)
(46,)


### Shape explanation
X is a 3D array with shape (n_trials, 2*n_channels, n_times) <br>

Y is a 1D array with shape (n_trials,)  Labels: 1 -> audio, 2 -> silence control, 4 -> video<br>

n_times = epoch duration * sampling rate = 18 s * 3.9 Hz <br>

n_ trial = 18 audio trials + 18 audio trials + 10 silence trials <br>

2*n_channels = We have 2 signals from each channel : HbO and HbR. The initial number of n_channels = 44. However, there are less number of channels because some optodes  were not well attached to the scalp, so they were rejected. 

## You can select only the HbO or Hbr channels

In [29]:
epochs.pick('hbo')
x_hbo = epochs.get_data()
y_hbr = epochs.events[:, 2]
print(x_hbo.shape)
print(y_hbr.shape)

(46, 38, 71)
(46,)


In [33]:
epochs.pick('hbr')
x_hbr = epochs.get_data()
y_hbr = epochs.events[:, 2]
print(x_hbr.shape)
print(y_hbr.shape)

(46, 38, 71)
(46,)


### You can also get the channel names 

In [34]:
ch_names = epochs.ch_names
print(ch_names)

['S1_D1 hbr', 'S2_D1 hbr', 'S3_D1 hbr', 'S3_D2 hbr', 'S4_D1 hbr', 'S4_D2 hbr', 'S5_D1 hbr', 'S5_D2 hbr', 'S5_D3 hbr', 'S6_D3 hbr', 'S6_D4 hbr', 'S7_D3 hbr', 'S7_D5 hbr', 'S7_D6 hbr', 'S8_D3 hbr', 'S8_D4 hbr', 'S8_D6 hbr', 'S8_D7 hbr', 'S9_D5 hbr', 'S9_D6 hbr', 'S9_D7 hbr', 'S9_D13 hbr', 'S10_D8 hbr', 'S10_D9 hbr', 'S11_D8 hbr', 'S11_D9 hbr', 'S11_D10 hbr', 'S11_D11 hbr', 'S11_D12 hbr', 'S12_D8 hbr', 'S12_D10 hbr', 'S12_D11 hbr', 'S13_D10 hbr', 'S13_D11 hbr', 'S13_D12 hbr', 'S14_D13 hbr', 'S14_D15 hbr', 'S15_D14 hbr']
